# M3L1 - RNA-Seq (Pseudo-Alignment) Overview

This Notebook provides an overview of pseudo-alignment techniques for mapping RNA-Seq reads to a transcriptome, as well as the data that will be processed this week using `salmon`. **This week in Module 3, we will perform a full pseudo-alignment pipeline for processing RNA-Seq data.**

**By the end of this notebook, you should be familiar with the following concepts:** 
- Psuedo-aligners for mapping RNA-Seq reads to a transcriptome, such as `salmon` and `kallisto`.
- The structure of the OPT3 Arabidopsis data set that will be processed using a pseudo-alignment pipeline.
- The structure of RNA-Seq pseudo-alignment outputs, count matrices, and their FKPM/TPM values

```{note}
The expectation-maximization walkthrough below (the toy read/transcript example and its figures) appears to follow teaching material by Alexander Lachmann. The original source and its license have not been confirmed yet, so the figures were removed from this public version.
```

## Alignment to the genome or transcriptome vs pseudo-alignment 


In RNA-Seq pipelines, there are multiple options to quantify gene expression from sample reads. Broadly, RNA-Seq aims to compare gene expression between sample conditions; thus, the mapping and quantification of these reads depend upon the quality of the alignment and the quality of our sample reads. Several tools have been developed to map reads to a genome or a transcriptome, and they vary in the amount of resources required to align reads to the reference file. 

- Traditionally, **genome alignment** is used to map reads individually to a reference genome. Genome alignment tools, such as `STAR` to `HISTAT2`, are very accurate. They provide genome alignments containing specific coordinates within the genome and alignment scores for each individual read, but this process is very computationally intensive. These genome alignments can be viewed within genome browsers and can be used for variant calling, splice detection, or the discovery of new transcripts. To quantify the reads, additional tools must be used to produce count matrices for downstream differential expression analysis. 

- Alternatively, **pseudo-aligners** map reads to known transcripts without computing the individual alignment of each base pair. These tools are light-weight and produce counts of the number of reads associated with each **known transcript** within a reference transcriptome, meaning that the mapping and quantification of reads are performed within a single step. 

| Feature                     | Traditional Alignment (STAR, HISAT2) | Pseudo-Alignment (Salmon, Kallisto) |
|-----------------------------|----------------------------------|--------------------------------------|
| Maps to                     | Genome/Transcriptome                          | Transcriptome                        |
| Output                      | BAM/SAM alignments              | Quantification tables (TPM, counts)  |
| Computationally Intensive?                    | Yes                        | No, can be performed within minutes                          |
| Can detect novel features?  | Yes                             | No                                   |
| Needs separate quant step?  | Yes (`featureCounts`, etc.)     | No (quant is built-in)               |
| Useful for                  | Investigating Splicing, variants, fusions     | Investigating differential expression   |

## Why are we using pseudo-alignment first?

The goal of this course is to educate students on the basics of processing and analyzing NGS data, which requires both computational and biological knowledge. Previously, we covered the usage of Linux systems, NGS sequencing, and quality control analysis for sample data. In the following weeks, we will compare pseudo-alignment and traditional alignment techniques for processing RNA-Seq data. It is important not to lose sight of the broader goals within our learning, so we will perform a pseudo-alignment and briefly discuss the analysis of processed data (performing differential expression on count matrices from the OPT3 sample data set using R). The pseudo-alignment techniques used will provide practice in executing pipelines and cover the basics of interpreting results from DE analysis. 

## The OPT3 data set

In Module 2 Exercise 2, we had performed fastp on 6 samples from [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157). These samples were *Arabidopsis thaliana* root tissues from two treatments within the [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157) data set, 3 biological replicates of the col-0 control, and 3 biological replicates of plants treated with roots treated with 20 µM CdCl₂ for 72 h. [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157) is a super-set(A culmination of multiple GSE data sets). 

**This week, we will perform pseudo-alignment on 6 samples from this same data set, but this time we will compare the col-0 roots and OPT3 mutants roots in replete conditions.**

Please review the video below for a brief description of the sample data set, as well as pseudo-alignment techniques:

<iframe width="560" height="315" src="https://www.youtube.com/embed/aobj5UUUXmg?si=_zaxlIVofwe370rX" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

**REQUIRED READINGS**: Please read through the two papers on Canvas, which are also included within the resources folder in Module 3. 
 - [Near-optimal probabilistic RNA-seq quantification ](https://www.nature.com/articles/nbt.3519)
 - [salmon provides fast and
bias-aware quantification
of transcript expression](https://obj.umiacs.umd.edu/salmon/nmeth.4197.pdf)

**NOTE:** There was a mistake in M2E2's labeling of the sample sets. The samples listed were the OPT3 controls rather than the Col-0 Cd 72hr treatment samples in M2E2 from the GSE set [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?). We are not going to utilize these samples from last week in this week's exercise, so there is no effect on the data processing for Module 3; however, please note the labeling difference in the table below: 

| GSM ID      | Sample ID                     | Genotype | Tissue | Treatment | Bio Rep | Tech Rep |
|-------------|-------------------------------|----------|--------|-----------|---------|----------|
| GSM2090459  | CRN1-1                        | Col-0    | Root   | Replete   | 1       | 1        |
| GSM2090462  | CRN2-1                        | Col-0    | Root   | Replete   | 2       | 1        |
| GSM2090465  | CRN3-1                        | Col-0    | Root   | Replete   | 3       | 1        |
| GSM2090468  | CLN1-1                        | Col-0    | Leaf   | Replete   | 1       | 1        |
| GSM2090471  | CLN2-1                        | Col-0    | Leaf   | Replete   | 2       | 1        |
| GSM2090474  | CLN3-1                        | Col-0    | Leaf   | Replete   | 3       | 1        |
| GSM2090477  | ORN1-1                        | OPT3     | Root   | Replete   | 1       | 1        |
| GSM2090480  | ORN2-1                        | OPT3     | Root   | Replete   | 2       | 1        |
| GSM2090483  | ORN3-1                        | OPT3     | Root   | Replete   | 3       | 1        |
| GSM2090486  | OLN1-1                        | OPT3     | Leaf   | Replete   | 1       | 1        |
| GSM2090489  | OLN2-1                        | OPT3     | Leaf   | Replete   | 2       | 1        |
| GSM2090492  | OLN3-1                        | OPT3     | Leaf   | Replete   | 3       | 1        |
| GSM3665895  | CST1-Col_shoot_treated_1-1    | Col-0    | Shoot  | Treated   | 1       | 1        |
| GSM3665898  | CST2-Col_shoot_treated_2-1    | Col-0    | Shoot  | Treated   | 2       | 1        |
| GSM3665901  | CST3-Col_shoot_treated_3-1    | Col-0    | Shoot  | Treated   | 3       | 1        |
| GSM3665904  | CRT1-Col_root_treated_1-1     | Col-0    | Root   | Treated   | 1       | 1        |
| GSM3665907  | CRT2-Col_root_treated_2-1     | Col-0    | Root   | Treated   | 2       | 1        |
| GSM3665910  | CRT3-Col_root_treated_3-1     | Col-0    | Root   | Treated   | 3       | 1        |
| GSM3665913  | OST1-opt3-2_shoot_treated_1-1 | OPT3     | Shoot  | Treated   | 1       | 1        |
| GSM3665916  | OST2-opt3-2_shoot_treated_2-1 | OPT3     | Shoot  | Treated   | 2       | 1        |
| GSM3665919  | OST3-opt3-2_shoot_treated_3-1 | OPT3     | Shoot  | Treated   | 3       | 1        |
| GSM3665922  | ORT1-opt3-2_root_treated_1-1  | OPT3     | Root   | Treated   | 1       | 1        |
| GSM3665925  | ORT2-opt3-2_root_treated_2-1  | OPT3     | Root   | Treated   | 2       | 1        |
| GSM3665928  | ORT3-opt3-2_root_treated_3-1  | OPT3     | Root   | Treated   | 3       | 1        |




## Steps in our RNA-Seq pseudo-alignment pipeline

As mentioned before, pseudo alignment is used by tools such as `salmon` and `kallito` to quantify transcript abundances directly from reads to the transcriptome without performing a traditional base-by-base sequence alignment.

In this module, we will take data from the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series containing samples of Col-0 and OPT3 mutant *Arabidopsis thaliana* root and shoot tissues produced in the experiments by Mendoza and McInturf. We will use 6 samples (CRN1-1, CRN2-1, CRN3-1, ORN1-1, ORN2-1, ORN3-1) to compare gene expression between the roots of Col-0 and the OPT3 mutants in replete conditions. 

**We know that the OPT3 mutant should express an iron deficiency response within the roots because a loss-of-function mutation in OPT3 in *Arabidopsis thaliana* leads to a constitutive iron deficiency response in the roots, even when sufficient iron is available. The OPT3 protein is crucial for transporting iron from the shoot to the roots, allowing the roots to "sense" the plant's overall iron sufficiency. Without functional OPT3, the root "thinks" the plant is deficient in iron and activates iron uptake pathways.** This was discussed in the paper covered in Module 2, [Cadmium interference with iron sensing reveals transcriptional programs sensitive and insensitive to reactive oxygen species](https://academic.oup.com/jxb/article/73/1/324/6367699). 

For this RNA-Seq pseudo-alignment pipeline, we will follow these steps: 
*[Third-party RNA-seq workflow figure omitted from the public version. A similar workflow figure is in the [HBC Intro-to-DGE lessons](https://hbctraining.github.io/Intro-to-DGE/).]*


1. Gather the `fastq` reads from each of the six single-end samples in our experiment (CRN1-1, CRN2-1, CRN3-1, ORN1-1, ORN2-1, ORN3-1) - **M2L2**
2. Run `fastp` so that the six samples are QC checked, trimmed, and QC checked after trimming - **M2P1**
3. Mapping to the transcriptome using `salmon` for our pseudo-alignment. This tool will produce transcript counts, which can be mapped directly to genes within `salmon`, producing our gene-level counts file for use in DE analysis - **M2P2**
4. Quality assessment of the gene-level count matrix produced, DE analysis, and post-processing to assign differentially expressed genes to their gene-ontology pathways - **M2E1**


Before we can continue running our pipeline, we should understand how pseudo-alignment works. 


**To perform a pseudo-alignment using `salmon` or `kallisto`, the following inputs are required:**
- Trimmed and QC checked `fastq` files containing the reads generated by sequencing
- Reference transcriptome (A `fasta` file containing transcript sequences)
- Annotated transcriptome (optional, but this `gff` file can be used by the aligner to convert transcript to gene counts)

The pseudo-alignment tool then performs these basic steps:

1. **Indexing the transcriptome**
   - An index of the transcritome is generated by separating transcripts into k-length sequences called k-mers and storing them in a large data structure, such as a hash table, so they can be referenced quickly. Both `salmon` and `kallisto` use k-mers to perform quick computations relative to traditional aligners like `STAR`.
   - This index contains the transcriptome segmented into short k-mers so that it can be compared to reads very quickly
2. **Producing k-mers from reads and matching them to k-mers in the transcriptome index.**
   - Short sequences from the read can be quickly referenced to the index of transcripts, rather than making sure each base is paired individually.
   - This produces a set of candidate transcripts where each read *could* have originated.
   - No true alignment is generated by these programs; they are checking if the k-mers in the read and the k-mers in the transcripts are similar. If the k-mers match, this read possibly generates this transcript.
3. **Quantification of transcripts from reads from their matching k-mers:**
   - Some transcripts can share k-mers, so some reads are ambiguous. This means the reads could have matched multiple transcripts.
   - These ambiguous reads are probabilistically assigned to transcripts based on overall compatibility and abundance. Expectation-maximization models are used to infer the most likely transcript abundances.
   - All of these resolved reads are counted and organized by transcripts to produce a count matrix describing transcript-level abundance of the reads from the sample.

**Outputs from the pseudo-alignment tools:**
- Transcript-level counts are produced (TPM, FKPM, raw counts)
- Gene-level summaries are produced from the annotated transcriptome. To produce gene-level counts, the `gff` annotation must be provided to `salmon` for use of the built-in gene model, or can be produced after gathering transcript counts with tools like `tximport` and a `tx2gene` table. 

## How do pseudo-alignment tools assign counts to transcripts if the k-mer aligns to multiple transcripts? Expectation maximization!

It is easy to understand how k-mers can be used to assign reads to transcripts conceptually; if the k-mers match, then it is possible that the read can be assigned to a specific transcript. In this example, we will refer to reads as a whole to simplify the concepts that pseudo-aligners use to match reads to their transcripts, but acknowledge that within the software, reads are split into multiple k-mers. 

Aligned reads can't easily be assigned to a transcript, as ambiguous assignments occur frequently. In some cases, only 30% of reads will uniquely align to one transcript. The main reason that this happens is that most genes have multiple transcripts that share exons. Secondly, genes from the same gene family often have similar sequences. Below is a figure depicting an example of a gene(`Exons`) and two transcripts. The reads, shown above the gene, from the sample show ambiguous and unique matches to both transcripts. 

*[Third-party figure omitted from the public version; original source not confirmed.]*


Reads in purple can come from either `transcript 1` or `transcript 2`. Some regions, however, are unique. This is true for unique splice junctions, where two exons are fused. **Expectation Maximization** takes reads and finds the most likely assignment for that read's transcript. There is a lot of math behind this, but we will stick to some simple examples that show how the solution is approximated. This approach can be further refined by adding additional normalization parameters such as transcript length and GC content.

If we were to count these reads without using expectation maximization, it is likely that the results of the pseudo-alignment would be useless in quantifying transcript abundance within our RNA-Seq experiments due to biases from transcript size, repeated use of exons within prominent gene families, or GC bias within sequencing.


### Naive Read Alignment
Let's look at an example of how we could assign these reads without taking into account their transcript abundance:

In this example, we will split the count attribution of a read between all the matching transcripts. **The image below shows one read(count) matching to three transcripts(1,2,3). In our naive alignment, each of the transcripts, 1, 2, and 3, would receive a third of a count.**

*[Third-party figure omitted from the public version; original source not confirmed.]*

In [ ]:
# observed alignments between the 4 reads (k-mers) and 6 transcripts(0, 1, 2, 3, 4, and 5)
aligned_reads = [[0,1], [0,3], [3,4,5], [4,5]]

# 6 total transcripts
counts = [0]*6

# We divide out each read to its assigned transcript. ie, read 3 pairs to 3 transcripts, so each gets .333 of a count.
for a in aligned_reads:
    for i in a:
        counts[i] = counts[i] + 1/len(a)
#print the counts for each transcript 1 through 6
print(counts)

#print the total counts, there should be 4 counts for four reads
sum(counts)

### Expectation Maximization

If we split the reads evenly, then it is likely that the resulting counts would be biased to transcripts sharing exons that are used frequently within gene families. In expectation maximization models, we don't split counts proportionally to the number of transcripts within the alignment. We split the counts based on the number of expected counts that a transcript would have. This weight, or probability, that a count will align to a transcript is relative to the number of counts that a transcript has. 
**In the figure below, one read(count) being split between the 3 transcripts based on the number of counts that each transcript already has.**
*[Third-party figure omitted from the public version; original source not confirmed.]*


To define the number of counts a transcript should have within our sample, outlining the count weight or probability, we must define the initial state of our transcripts. In this example, all transcripts start off having the same probability counts = [1]*6. None of the initial values should be 0 (they will stay 0 otherwise).

We will now iteratively update the probabilities that a count will match to this transcript (In this example, 100 times), by looping over the alignments and splitting counts relative to their previous probabilities. After each iteration, we replace the transcript probabilities with the newly found counts (probabilities do not need to be normalized).

We iterate until there are only very small changes in our probabilities, and we have converged to a global maximum (it can be proven that the solution is not local but global). This means that we can now assign weight to each count as it is aligned to transcripts, relative to the transcripts' abundance within the original reads from our sample. 

In [ ]:
# observed alignments between the 4 reads (k-mers) and 6 transcripts(0, 1, 2, 3, 4, and 5)
aligned_reads = [[0,1], [0,3], [3,4,5], [4,5]]

counts = [1]*6
temp_counts = [0]*6
intermediate_counts = []

for k in range(100):
    temp_counts = [0]*6
    for a in aligned_reads:

        sum_counts = 0
        for i in a:
            sum_counts = sum_counts + counts[i]

        for i in a:
            temp_counts[i] = temp_counts[i] + counts[i]/sum_counts
    counts = temp_counts.copy()
    intermediate_counts.append(counts)

print(counts)
sum(counts)

 A notebook by Alexander Lachmann (not included in this repository) investigates the basics of these EM models, accounting for transcript length and the GC bias within Illumina Sequencing. If you have extra time or are interested, please take a look. Even within EM models, there are biases that arise. In the case that a transcript is unambiguous, there will be a much higher weight assigned to the probability of said transcript. The length of transcripts will affect the number of reads that are mapped to a certain transcript, meaning longer transcripts will result in more mapped reads. The EM models used by `salmon` and `kallisto` are complex and account for many of these factors, so they will not be discussed in detail within this course. 

## pseudo-alignment outputs: count matrices and TPM/FKPM values
After performing pseudo-alignment with tools like `salmon` or `kallisto`, the output generally consists of files that summarize how many reads are assigned to each transcript or gene. These quantification outputs are crucial for downstream analyses like **differential expression**. 

It is important to note that pseudo-aligners generate counts at the transcript level. Many tools that analyze differential expression, such as `DESeq2`, utilize gene-level counts. This means that the pseudo-alignment file generated must be converted to gene-level counts for DE analysis, and also that genes that are not assigned transcripts will not be quantified. In some organisms, such as Arabidopsis thaliana, the transcriptome is well documented. This makes A. thaliana a good candidate for pseudo-alignment; however, new isoforms will not be discovered using these techniques. If you are working in an organism that has only recently undergone whole genome sequencing, it is better to use traditional alignment techniques.

### Outputs from pseudo-aligners
Both `salmon` and `kallisto` produce a file called `quant.sf` (Salmon) or `abundance.tsv` (Kallisto). These files are the transcript counts and contain:

| Column Name     | Description |
|-----------------|-------------|
| **Name**        | Transcript ID (e.g., ENST00000335137) |
| **Length**      | Length of the transcript (in nucleotides) |
| **EffectiveLength** | Adjusted length used for normalization (accounts for read length) |
| **TPM**         | *Transcripts Per Million* normalized per base and per sample |
| **NumReads**    | Estimated number of reads originating from the transcript |

### Count matrices 
A **count matrix** is a table where rows are genes/transcripts, and columns are samples. The values in the matrix represent the **raw or estimated counts** of sequencing reads assigned to each transcript. The `quant.sf` file produced by salmon contains these counts within a proprietary format. We will use this `quant.sf` file to perform our DE analysis.

Example structure:

| Transcript ID | Sample 1 | Sample 2 | Sample 3 |
|---------------|----------|----------|----------|
| ENST000001    | 1234     | 1456     | 1080     |
| ENST000002    | 567      | 610      | 544      |
| ...           | ...      | ...      | ...      |


Tools like `tximport` in R can convert the transcript-level counts from the `quant.sf` or `abundance.tsv` file to gene-level counts for use in differential expression analysis tools, such as `edgeR` or `DESeq2`.

### Raw counts vs TPM vs FPKM

Counts produced from alignments are dependent upon the transcript length and sequencing depth. We cannot use raw counts for our analysis for several reasons, but be aware that longer transcripts will have higher counts due to more reads aligning with them, and that sequencing depth will change the total number of reads as well. If we were to compare raw counts within a single sample across genes, we would see higher counts for longer transcripts. If we were to compare counts for the same gene across two samples, the counts could differ for many reasons, such as sequencing depth, composition bias, and confounding factors within the experiment. 

To make counts comparable, we **normalize** them. There are several methods for normalization of counts, such as TPM and FKPM. RPKM and TPM represent the relative abundance of transcripts in a sample but do not normalize for global shifts in total RNA content across samples. 

**FKPM (Fragments per kilobase per Million mapped reads)**
FKPM values normalize for transcript length relative to the total number of reads within a sample. 

`FKPM = (Read Count / Transcript length in Kb) / (Total number of reads in millions)`

**TPM (Transcripts per million)**
TPM improves on FPKM by flipping the order of the steps so that TPM values are comparable across samples.

`TPM = (Read Count / Transcript Length in Kb) ÷ (Sum of Length-Normalized Counts) × 1,000,000`

IE, for a given RNA sample, if you were to sequence one million full-length transcripts, a TPM value represents the number of transcripts you would have seen for a given gene or isoform. The average TPM is equal to 10^6 (1 million) divided by the number of annotated transcripts in a given annotation, meaning that TPM values act as a constant. 

To learn more about TPM/FKPM normalization, check out this article:  [Misuse of RPKM or TPM normalization when comparing  across samples and sequencing protocols](https://pubmed.ncbi.nlm.nih.gov/32284352/)

